In [2]:
import numpy as np
import pandas as pd

def generate_metabolic_cohort(n_samples: int = 200, seed: int = 42) -> pd.DataFrame:
    """
    Generates a synthetic baseline cohort with age-driven trajectories 
    and realistic clinical noise parameters for metabolic biomarkers.
    """
    np.random.seed(seed)
    mock_age = np.random.uniform(20, 80, n_samples)
    
    # Biomarker simulation based on clinical longevity targets/slopes + Gaussian noise (epsilon)
    # Fasting Insulin (µIU/mL): baseline ~4.0, rises slightly with age + noise SD 3.5
    insulin = 4.0 + 0.06 * mock_age + np.random.normal(0, 3.5, n_samples)
    
    # hs-CRP (mg/L): baseline ~0.5, mild inflammatory increase with age + noise SD 0.8
    crp = 0.5 + 0.015 * mock_age + np.random.normal(0, 0.8, n_samples)
    
    # Fasting Glucose (mg/dL): baseline ~82, minor upward drift with age + noise SD 7.0
    glucose = 82.0 + 0.18 * mock_age + np.random.normal(0, 7.0, n_samples)
    
    # HbA1c (%): baseline ~5.1%, gradual non-enzymatic glycation increase + noise SD 0.3
    hba1c = 5.1 + 0.005 * mock_age + np.random.normal(0, 0.3, n_samples)
    
    # Ensure clinical floor constraints (values cannot realistically drop below absolute minimums)
    insulin = np.clip(insulin, 1.0, 50.0)
    crp = np.clip(crp, 0.1, 20.0)
    glucose = np.clip(glucose, 50.0, 200.0)
    hba1c = np.clip(hba1c, 4.0, 14.0)
    
    df = pd.DataFrame({
        'age': mock_age,
        'insulin': insulin,
        'crp': crp,
        'glucose': glucose,
        'hba1c': hba1c
    })
    df.to_csv("sinthetic_data_set.csv")
    return df

def calculate_homa_ir(df: pd.DataFrame, 
                      insulin_col: str = 'insulin', 
                      glucose_col: str = 'glucose', 
                      is_si_units: bool = False) -> pd.DataFrame:
    """
    Computes HOMA-IR from paired insulin and glucose columns.
    - Standard US units: Glucose in mg/dL, Insulin in µIU/mL -> divisor is 405
    - SI metric units: Glucose in mmol/L, Insulin in µIU/mL -> divisor is 22.5
    """
    divisor = 22.5 if is_si_units else 405.0
    df = df.copy()
    df['homa_ir'] = (df[glucose_col] * df[insulin_col]) / divisor
    
    return df

# ==========================================
# Execution Example
# ==========================================
if __name__ == "__main__":
    # 1. Simulate data using metabolic baselines
    cohort_df = generate_metabolic_cohort(n_samples=1000)
    
    
    # 2. Calculate HOMA-IR values
    analyzed_df = calculate_homa_ir(cohort_df)
    
    print("--- Simulated Cohort with HOMA-IR ---")
    print(analyzed_df[['age', 'glucose', 'insulin', 'homa_ir']].to_string(index=False))


--- Simulated Cohort with HOMA-IR ---
      age    glucose   insulin  homa_ir
42.472407  95.766653  7.170298 1.695495
77.042858  91.319358  3.948866 0.890390
63.919637  85.083129  9.165871 1.925583
55.919509  84.771201  9.492221 1.986832
29.361118  83.874581  7.720934 1.598988
29.359671  84.821292  9.544313 1.998916
23.485017  80.837331  8.327829 1.662221
71.970569  85.881886  9.925364 2.104714
56.066901  88.899196  7.118434 1.562526
62.484355  94.517170  1.935698 0.451745
21.235070  89.987519  6.777768 1.505962
78.194591  92.173065  9.418582 2.143555
69.946558  91.699396  9.147319 2.071120
32.740347  81.406374  1.495801 0.300661
30.909498  87.343635  2.070872 0.446611
31.004271  81.641768  9.546291 1.924385
38.254535  92.897711  6.156829 1.412235
51.485386  78.766304  9.474376 1.842621
45.916701  87.747598  6.854116 1.485018
37.473748  90.853026  6.352571 1.425063
56.711174  93.494853 10.686664 2.467032
28.369632 105.957769  3.896021 1.019293
37.528679  91.203762  6.588143 1.483614
41